# Territorial Digital Divide – Geospatial Raster Analysis
**Cusco, Peru | NASA VNL × Mobile Coverage**

Pipeline measuring digital inequality using nighttime radiance and mobile network density as proxies for urbanization and internet access.

---
## Setup

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter
from scipy import stats
import seaborn as sns
import pandas as pd
from pathlib import Path

# Paths
DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

VNL_PATH  = DATA_DIR / 'VNL_cusco_2025.tif'
KERN_PATH = DATA_DIR / 'kernel_cobmovil2019_50m.tif'

---
## Steps 0–1 | Load & Inspect Rasters

In [ ]:
def inspect_raster(path: Path, label: str) -> dict:
    """Open a raster and print its key metadata."""
    with rasterio.open(path) as src:
        info = {
            'label':      label,
            'crs':        src.crs.to_string(),
            'width':      src.width,
            'height':     src.height,
            'count':      src.count,
            'nodata':     src.nodata,
            'bounds':     src.bounds,
            'res_x':      src.res[0],
            'res_y':      src.res[1],
            'dtype':      src.dtypes[0],
        }
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  CRS          : {info['crs']}")
    print(f"  Dimensions   : {info['height']} rows × {info['width']} cols")
    print(f"  Bands        : {info['count']}")
    print(f"  Data type    : {info['dtype']}")
    print(f"  NoData value : {info['nodata']}")
    print(f"  Bounds       : {info['bounds']}")
    print(f"  Pixel res    : {info['res_x']:.6f} × {info['res_y']:.6f} (x, y)")
    return info

vnl_info  = inspect_raster(VNL_PATH,  'VNL_cusco_2025.tif  (NASA Nighttime Radiance)')
kern_info = inspect_raster(KERN_PATH, 'kernel_cobmovil2019_50m.tif  (Mobile Coverage Kernel)')

---
## Step 2 | Reproject & Align Connectivity to VNL Grid

In [ ]:
with rasterio.open(VNL_PATH) as vnl_src:
    vnl_meta    = vnl_src.meta.copy()
    vnl_data    = vnl_src.read(1).astype(np.float32)
    vnl_nodata  = vnl_src.nodata
    vnl_crs     = vnl_src.crs
    vnl_transform = vnl_src.transform

with rasterio.open(KERN_PATH) as kern_src:
    kern_crs = kern_src.crs

    # Calculate transform from UTM-19S → WGS84 with VNL dimensions
    dst_transform, dst_width, dst_height = calculate_default_transform(
        kern_crs, vnl_crs,
        kern_src.width, kern_src.height,
        *kern_src.bounds
    )

    # Reproject connectivity to a temporary array in WGS84
    kern_reprojected = np.zeros((vnl_data.shape[0], vnl_data.shape[1]), dtype=np.float32)

    reproject(
        source=rasterio.band(kern_src, 1),
        destination=kern_reprojected,
        src_transform=kern_src.transform,
        src_crs=kern_crs,
        dst_transform=vnl_transform,   # align to VNL grid
        dst_crs=vnl_crs,
        resampling=Resampling.bilinear
    )

print(f"VNL shape           : {vnl_data.shape}")
print(f"Connectivity shape  : {kern_reprojected.shape}")
print(f"Grids aligned       : {vnl_data.shape == kern_reprojected.shape}")

---
## Step 3 | Percentile Normalization [0, 1]

In [ ]:
def normalize_percentile(arr: np.ndarray, nodata=None, p_low=2, p_high=98) -> np.ndarray:
    """Clip to [p_low, p_high] percentile and scale to [0, 1]. NoData and negatives → 0."""
    out = arr.copy().astype(np.float32)

    # Mask nodata and negatives
    if nodata is not None:
        out[out == nodata] = np.nan
    out[out < 0] = np.nan

    valid = out[~np.isnan(out)]
    lo, hi = np.percentile(valid, p_low), np.percentile(valid, p_high)
    print(f"  p{p_low}={lo:.4f}  p{p_high}={hi:.4f}")

    out = np.clip(out, lo, hi)
    out = (out - lo) / (hi - lo)
    out = np.nan_to_num(out, nan=0.0)   # replace remaining NaN → 0
    return out

print("VNL normalization:")
vnl_norm = normalize_percentile(vnl_data, nodata=vnl_nodata)

print("Connectivity normalization:")
conn_norm = normalize_percentile(kern_reprojected)

print(f"\nVNL  norm range  : [{vnl_norm.min():.4f}, {vnl_norm.max():.4f}]")
print(f"Conn norm range  : [{conn_norm.min():.4f}, {conn_norm.max():.4f}]")

---
## Step 4 | VNL Map: Raw vs. Normalized

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw VNL (mask nodata)
raw_display = vnl_data.copy().astype(np.float32)
if vnl_nodata is not None:
    raw_display[raw_display == vnl_nodata] = np.nan
raw_display[raw_display < 0] = np.nan

im0 = axes[0].imshow(raw_display, cmap='inferno', interpolation='nearest')
axes[0].set_title('VNL Raw — Nighttime Radiance', fontsize=13)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='Radiance (nW/cm²/sr)')

im1 = axes[1].imshow(vnl_norm, cmap='inferno', vmin=0, vmax=1, interpolation='nearest')
axes[1].set_title('VNL Normalized [0–1]', fontsize=13)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label='Normalized radiance')

fig.suptitle('NASA Black Marble — Cusco 2025', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 5 | IBD & EDT Indices

In [ ]:
# IBD: Digital Divide Index  →  range [-1, 1]
# Positive = more light than connectivity (divide); negative = more coverage than light
ibd = vnl_norm - conn_norm

# EDT: Total Digital Exclusion  →  range [0, 1]
# High = dark AND no coverage (severe exclusion)
edt = (1 - vnl_norm) * (1 - conn_norm)

print(f"IBD range : [{ibd.min():.4f}, {ibd.max():.4f}]")
print(f"EDT range : [{edt.min():.4f}, {edt.max():.4f}]")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(ibd, cmap='RdBu_r', vmin=-1, vmax=1, interpolation='nearest')
axes[0].set_title('IBD — Digital Divide Index\n(+) more light than coverage | (−) more coverage than light', fontsize=11)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(edt, cmap='YlOrRd', vmin=0, vmax=1, interpolation='nearest')
axes[1].set_title('EDT — Total Digital Exclusion\nHigh = dark AND no mobile coverage', fontsize=11)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.suptitle('Digital Divide Indices — Cusco 2025', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6 | Intervention Priority Classification (3 Levels)

In [ ]:
# Thresholds
VNL_THRESH  = 0.2   # below = low urbanization
CONN_THRESH = 0.2   # below = low connectivity

# Priority levels:
# 3 = High   : low VNL AND low connectivity  (most urgent)
# 2 = Medium : low VNL OR  low connectivity  (partial exclusion)
# 1 = Low    : neither condition             (well-served)
low_vnl  = vnl_norm  < VNL_THRESH
low_conn = conn_norm < CONN_THRESH

priority = np.ones(vnl_norm.shape, dtype=np.uint8)   # default: Low
priority[low_vnl | low_conn]   = 2                   # Medium
priority[low_vnl & low_conn]   = 3                   # High

# Statistics table
labels = {1: 'Low priority', 2: 'Medium priority', 3: 'High priority'}
total_px = priority.size
rows = []
for lvl, name in labels.items():
    count = int((priority == lvl).sum())
    rows.append({'Level': lvl, 'Label': name, 'Pixels': count, 'Pct (%)': round(count / total_px * 100, 2)})
df_priority = pd.DataFrame(rows)
print(df_priority.to_string(index=False))

cmap_p = mcolors.ListedColormap(['#2ecc71', '#f39c12', '#e74c3c'])
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(priority, cmap=cmap_p, vmin=1, vmax=3, interpolation='nearest')
legend_elements = [Patch(facecolor='#2ecc71', label='Low priority'),
                   Patch(facecolor='#f39c12', label='Medium priority'),
                   Patch(facecolor='#e74c3c', label='High priority')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.set_title('Intervention Priority — Cusco 2025', fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Step 7 | Social Exclusion Risk + Gaussian Smoothing (σ=5)

In [ ]:
risk_raw    = edt * (1 - vnl_norm)
risk_smooth = gaussian_filter(risk_raw, sigma=5)

print(f"Risk raw    range : [{risk_raw.min():.4f}, {risk_raw.max():.4f}]")
print(f"Risk smooth range : [{risk_smooth.min():.4f}, {risk_smooth.max():.4f}]")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(risk_raw, cmap='hot_r', vmin=0, vmax=1, interpolation='nearest')
axes[0].set_title('Social Exclusion Risk (raw)', fontsize=12)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(risk_smooth, cmap='hot_r', vmin=0, vmax=1, interpolation='nearest')
axes[1].set_title('Social Exclusion Risk (Gaussian σ=5)', fontsize=12)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.suptitle('Social Exclusion Risk — Cusco 2025', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 8 | Territorial 2×2 Classification + GeoTIFF Export

In [ ]:
# 2×2 matrix: VNL high/low × Connectivity high/low
# Class 1 — Urban Connected   : VNL ≥ 0.3 AND conn ≥ 0.3
# Class 2 — Urban Divide      : VNL ≥ 0.3 AND conn < 0.3
# Class 3 — Rural Connected   : VNL < 0.3  AND conn ≥ 0.3
# Class 4 — Critical Divide   : VNL < 0.3  AND conn < 0.3  (max exclusion)
TH = 0.3
high_vnl  = vnl_norm  >= TH
high_conn = conn_norm >= TH

terr_class = np.zeros(vnl_norm.shape, dtype=np.uint8)
terr_class[ high_vnl &  high_conn] = 1
terr_class[ high_vnl & ~high_conn] = 2
terr_class[~high_vnl &  high_conn] = 3
terr_class[~high_vnl & ~high_conn] = 4

# Statistics DataFrame
class_labels = {
    1: 'Urban Connected',
    2: 'Urban Divide',
    3: 'Rural Connected',
    4: 'Critical Divide'
}
rows = []
for cls, name in class_labels.items():
    mask = terr_class == cls
    rows.append({
        'Class': cls,
        'Label': name,
        'Pixels': int(mask.sum()),
        'Pct (%)': round(mask.sum() / terr_class.size * 100, 2),
        'Mean VNL': round(float(vnl_norm[mask].mean()) if mask.any() else 0, 4),
        'Mean Conn': round(float(conn_norm[mask].mean()) if mask.any() else 0, 4),
    })
df_terr = pd.DataFrame(rows)
print(df_terr.to_string(index=False))

# Export GeoTIFF
out_path = OUTPUT_DIR / 'territorial_classification.tif'
meta = vnl_meta.copy()
meta.update({'dtype': 'uint8', 'count': 1, 'nodata': 0})
with rasterio.open(out_path, 'w', **meta) as dst:
    dst.write(terr_class, 1)
print(f"\nSaved → {out_path}")

# Map
cmap_t = mcolors.ListedColormap(['#3498db', '#e67e22', '#27ae60', '#c0392b'])
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(terr_class, cmap=cmap_t, vmin=1, vmax=4, interpolation='nearest')
legend_elements = [
    Patch(facecolor='#3498db', label='1 — Urban Connected'),
    Patch(facecolor='#e67e22', label='2 — Urban Divide'),
    Patch(facecolor='#27ae60', label='3 — Rural Connected'),
    Patch(facecolor='#c0392b', label='4 — Critical Divide'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.set_title('Territorial Classification — Cusco 2025', fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Step 9 | Statistical Analysis

In [ ]:
# --- Descriptive statistics by class ---
desc_rows = []
for cls, name in class_labels.items():
    mask = terr_class == cls
    if not mask.any():
        continue
    v = vnl_norm[mask]
    c = conn_norm[mask]
    desc_rows.append({
        'Class': f"{cls} — {name}",
        'VNL mean': round(v.mean(), 4), 'VNL std': round(v.std(), 4),
        'VNL p25': round(np.percentile(v, 25), 4), 'VNL p75': round(np.percentile(v, 75), 4),
        'Conn mean': round(c.mean(), 4), 'Conn std': round(c.std(), 4),
    })
df_desc = pd.DataFrame(desc_rows)
print("Descriptive statistics by class:")
print(df_desc.to_string(index=False))

# --- Pearson correlation (40-pixel subsample) ---
rng = np.random.default_rng(42)
flat_vnl  = vnl_norm.ravel()
flat_conn = conn_norm.ravel()
idx = rng.choice(len(flat_vnl), size=40, replace=False)
r, p = stats.pearsonr(flat_vnl[idx], flat_conn[idx])
print(f"\nPearson r (n=40 subsample): r={r:.4f}  p={p:.4f}")

# --- KDE plots ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = ['#3498db', '#e67e22', '#27ae60', '#c0392b']
for cls, name in class_labels.items():
    mask = terr_class == cls
    if not mask.any():
        continue
    c = colors[cls - 1]
    sns.kdeplot(vnl_norm[mask],  ax=axes[0], label=f"{cls} — {name}", color=c)
    sns.kdeplot(conn_norm[mask], ax=axes[1], label=f"{cls} — {name}", color=c)
axes[0].set_title('KDE — VNL Normalized by Class');  axes[0].set_xlabel('VNL norm')
axes[1].set_title('KDE — Connectivity by Class');    axes[1].set_xlabel('Connectivity norm')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# --- Welch's t-test: Class 1 vs Class 4 ---
cls1_vnl = vnl_norm[terr_class == 1].ravel()
cls4_vnl = vnl_norm[terr_class == 4].ravel()
t_stat, p_val = stats.ttest_ind(cls1_vnl, cls4_vnl, equal_var=False)
pooled_std = np.sqrt((cls1_vnl.std()**2 + cls4_vnl.std()**2) / 2)
cohen_d = (cls1_vnl.mean() - cls4_vnl.mean()) / pooled_std
print(f"\nWelch's t-test  (Class 1 vs Class 4, VNL norm):")
print(f"  t-statistic : {t_stat:.4f}")
print(f"  p-value     : {p_val:.6f}")
print(f"  Cohen's d   : {cohen_d:.4f}")

---
## Step 10 | Export Rasters + Composite Dashboard

In [ ]:
# Helper: save float32 raster with same CRS/transform as VNL
def save_raster(arr: np.ndarray, filename: str, nodata: float = -9999.0):
    meta = vnl_meta.copy()
    meta.update({'dtype': 'float32', 'count': 1, 'nodata': nodata})
    path = OUTPUT_DIR / filename
    with rasterio.open(path, 'w', **meta) as dst:
        dst.write(arr.astype(np.float32), 1)
    print(f"Saved → {path}")

save_raster(vnl_norm,  'vnl_normalized.tif')
save_raster(conn_norm, 'connectivity_normalized.tif')
save_raster(ibd,       'ibd_index.tif')
# territorial_classification.tif already saved in Step 8

# --- Composite Dashboard ---
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Territorial Digital Divide — Cusco 2025', fontsize=16, fontweight='bold')

panels = [
    (axes[0, 0], vnl_norm,       'inferno',   0, 1,    'VNL Normalized'),
    (axes[0, 1], conn_norm,      'viridis',   0, 1,    'Connectivity Normalized'),
    (axes[0, 2], ibd,            'RdBu_r',   -1, 1,    'IBD — Digital Divide Index'),
    (axes[1, 0], edt,            'YlOrRd',    0, 1,    'EDT — Total Exclusion'),
    (axes[1, 1], risk_smooth,    'hot_r',     0, 1,    'Social Exclusion Risk (smoothed)'),
]
for ax, data, cmap, vmin, vmax, title in panels:
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Territorial classification
cmap_t = mcolors.ListedColormap(['#3498db', '#e67e22', '#27ae60', '#c0392b'])
im_t = axes[1, 2].imshow(terr_class, cmap=cmap_t, vmin=1, vmax=4, interpolation='nearest')
axes[1, 2].set_title('Territorial Classification', fontsize=10)
axes[1, 2].axis('off')
legend_elements = [
    Patch(facecolor='#3498db', label='1 Urban Connected'),
    Patch(facecolor='#e67e22', label='2 Urban Divide'),
    Patch(facecolor='#27ae60', label='3 Rural Connected'),
    Patch(facecolor='#c0392b', label='4 Critical Divide'),
]
axes[1, 2].legend(handles=legend_elements, loc='lower right', fontsize=7)

plt.tight_layout()
dashboard_path = OUTPUT_DIR / 'dashboard.png'
fig.savefig(dashboard_path, dpi=150, bbox_inches='tight')
print(f"Dashboard saved → {dashboard_path}")
plt.show()